# Academic Paper Knowledge Mapper: Project Motifs

This notebook demonstrates the core logic and algorithmic "motifs" of the Academic Paper Knowledge Mapper project. It covers the end-to-end pipeline from PDF parsing to interactive clustering and RAG-based questioning.

### Core Tech Stack:
- **Parsing**: PyPDF2
- **Clustering**: Scikit-Learn (TF-IDF, KMeans, PCA)
- **Vector Search**: FAISS & Sentence-Transformers
- **Visualization**: Matplotlib / Plotly

In [ ]:
# !pip install PyPDF2 faiss-cpu sentence-transformers scikit-learn matplotlib numpy

## Motif 1: Document Digestion & Chunking
We extract text from PDFs and split it into semantic chunks. This allows the LLM to process large documents within its context window while maintaining semantic meaning.

In [ ]:
import io
def chunk_text(text, chunk_size=1000, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return [c.strip() for c in chunks if len(c) > 50]

# Sample Text Demonstration
sample_doc = """Academic papers are complex documents. They often contain an abstract, introduction, 
methodology, results, and discussion. Mapping this knowledge requires understanding the relationship 
between different sections. Machine learning can help cluster similar topics together for better discovery."""

chunks = chunk_text(sample_doc, chunk_size=100, overlap=20)
print(f"Generated {len(chunks)} chunks.")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i}: {chunk[:50]}...")

## Motif 2: Knowledge Mapping (The ML Engine)
We use TF-IDF and K-Means to identify clusters of knowledge in the document. PCA is used to project these clusters into a 2D space for visualization.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

# 1. Vectorize chunks
vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(chunks)

# 2. Cluster using KMeans
n_clusters = 2
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
labels = kmeans.fit_predict(X)

# 3. Reduce to 2D for Visualization
pca = PCA(n_components=2)
coords = pca.fit_transform(X.toarray())

# 4. Plot (Motif of the KDMap UI)
plt.figure(figsize=(8, 6))
for i in range(n_clusters):
    cluster_pts = coords[labels == i]
    plt.scatter(cluster_pts[:, 0], cluster_pts[:, 1], label=f'Topic {i}')

plt.title("Knowledge Map Distribution (PCA)")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## Motif 3: RAG Pipeline & Vector Search
We use a local embedding model to vectorize chunks and store them in a FAISS index for high-speed retrieval. This allows the system to find the most relevant context for any user query.

In [ ]:
# This motif represents the core logic in rag_pipeline.py
from sentence_transformers import SentenceTransformer
import faiss

# Load Model (Small MiniLM for efficiency)
model = SentenceTransformer('all-MiniLM-L6-v2')

# Embed Chunks
embeddings = model.encode(chunks)
dim = embeddings.shape[1]

# Create FAISS Index
index = faiss.IndexFlatL2(dim)
index.add(np.array(embeddings).astype('float32'))

# Query Example
query = "How can machine learning help?"
query_embedding = model.encode([query])
distances, indices = index.search(np.array(query_embedding).astype('float32'), k=1)

print(f"Query: {query}")
print(f"Nearest Chunk: {chunks[indices[0][0]]}")

## Motif 4: Strict Hallucination-Free Prompting
The final piece of the motif is the specialized prompt that restricts the LLM to only answer based on the retrieved context. This is crucial for academic research to ensure accuracy.

```python
STRICT_PROMPT = """
You are a strict academic assistant.
Given ONLY the context provided below, answer the user's query.
IF the answer is not in the context, say: 'The answer is not available in the provided document.'
DO NOT use your prior knowledge.
---------------------
Context: {retrieved_context}
---------------------
Query: {user_query}
Answer:
"""
```

## Conclusion
These motifs form the backbone of the **Academic Paper Knowledge Mapper**. By combining semantic retrieval with structural visualization, we turn unstructured PDFs into interactive, queryable data landscapes.